# Laterality of HCP Task Maps
**HCP Workshop — hands-on notebook**

**Plan:** open the maps → pick Language & Social → split left / right → **Left − Right** → save.

*Requirements:* `pip install nibabel nilearn`.

## 1. Open the task maps
A CIFTI file is a table: **rows = maps** (task contrasts), **columns = brain locations**.

In [ ]:
import numpy as np
import nibabel as nib
from nilearn import plotting

folder = "HCP_S1200_GroupAvg_v1/"
file = folder + "HCP_S1200_997_tfMRI_ALLTASKS_level2_cohensd_hp200_s2_MSMAll.dscalar.nii"

img = nib.load(file)
data = img.get_fdata()
names = list(img.header.get_axis(0).name)
names

## 2. Pick the Language and Social maps

In [ ]:
language = names.index("tfMRI_LANGUAGE_STORY")
social = names.index("tfMRI_SOCIAL_TOM-RANDOM")

maps = data[[language, social]]

## 3. Split into left and right hemispheres
Each column is labelled with its hemisphere and vertex number. We put each hemisphere on its full surface of 32,492 vertices (`NaN` = no data, e.g. the medial wall).

In [ ]:
brain = img.header.get_axis(1)
is_left = brain.name == "CIFTI_STRUCTURE_CORTEX_LEFT"
is_right = brain.name == "CIFTI_STRUCTURE_CORTEX_RIGHT"

left = np.full((2, 32492), np.nan)
right = np.full((2, 32492), np.nan)

left[:, brain.vertex[is_left]] = maps[:, is_left]
right[:, brain.vertex[is_right]] = maps[:, is_right]

## 4. Left − Right
Vertex *i* on the left mirrors vertex *i* on the right, so we simply subtract.  
**Positive** = stronger on the left, **negative** = stronger on the right.

In [ ]:
laterality = left - right

## 5. Save
The saved file opens in **wb_view** on the left surface.

In [ ]:
language_map = nib.gifti.GiftiDataArray(laterality[0].astype("float32"))
social_map = nib.gifti.GiftiDataArray(laterality[1].astype("float32"))

gifti = nib.GiftiImage(darrays=[language_map, social_map])
nib.save(gifti, "Laterality_L-R.func.gii")